# Notebook 1 – Cài đặt SIFT từ đầu, minh họa từng bước

SIFT (Scale-Invariant Feature Transform) tìm các **điểm đặc trưng** và tạo **descriptor 128 chiều** cho mỗi điểm. Mọi phép xử lý ảnh trong notebook đều gọi hàm **tự viết** trong `python/image_ops.py` và `python/sift_extractor.py`.

| Bước | Nội dung |
|---|---|
| 1 | Không gian tỉ lệ: thu nhỏ ảnh (octave) + làm mờ Gaussian |
| 2 | DoG – hiệu 2 ảnh mờ liền kề |
| 3 | Tìm keypoint: cực trị so với 26 điểm lân cận |
| 4 | Lọc keypoint nằm trên cạnh (Hessian) |
| 5 | Gán hướng chính (histogram 36 bin) |
| 6 | Descriptor 128 chiều |
| 7 | Kiểm tra bất biến tỉ lệ và phép xoay |

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("python"))   # notebook đặt ở thư mục gốc repo
import numpy as np
import matplotlib.pyplot as plt
import image_ops as ops
from sift_extractor import SIFT
from panorama_stitching import Panorama
from run_panorama import read_rgb              # chỉ dùng OpenCV để ĐỌC file ảnh
from general_functions import show_images, normalize_img
%matplotlib inline

In [ ]:
img_rgb = read_rgb("images/img3.jpg")
img = normalize_img(ops.rgb2gray(img_rgb))     # ảnh xám, giá trị [0, 1]
show_images([img_rgb, img], ["Ảnh gốc", "Ảnh xám"], figsize=(12, 4))

## Bước 1 – Không gian tỉ lệ
- **Octave**: các phiên bản ảnh có kích thước khác nhau (hệ số 1, √2, 2, …) – `ops.resize_bilinear`.
- Trong mỗi octave làm mờ Gaussian với 6 mức sigma: σ, kσ, k²σ, … – `ops.gaussian_blur`.

In [ ]:
sift = SIFT(num_downsamples=3)
octaves = sift.downsample_img(img)
show_images(octaves, [f"Octave {i}: {o.shape[1]}x{o.shape[0]}" for i, o in enumerate(octaves)], figsize=(16, 4))
blurred = sift.create_octave(octaves[0])
show_images(blurred, [f"sigma = {s:.2f}" for s in sift.sigmas], figsize=(18, 3))

## Bước 2 – Difference of Gaussians (DoG)
`DoG_i = ảnh_mờ_i − ảnh_mờ_(i−1)`: xấp xỉ Laplacian of Gaussian, làm nổi bật đốm và góc.

In [ ]:
doG = sift.get_doG(blurred)
show_images([normalize_img(d) for d in doG], [f"DoG {i}" for i in range(len(doG))], figsize=(18, 3))

## Bước 3 & 4 – Tìm keypoint và lọc điểm nằm trên cạnh
- Cực trị: lớn nhất (hoặc nhỏ nhất) so với 8 điểm cùng ảnh + 9 điểm DoG trên + 9 điểm DoG dưới, và `|DoG| > 0.003`.
- Lọc: giữ điểm có `Tr(H)² / Det(H) < (r+1)²/r` với r = 10.

In [ ]:
doG_i = doG[1]
keypoints = sift.get_keypoints(doG[0:3], sift.doG_thresh)
dx, dy = ops.sobel(doG_i)
dxx, _ = ops.sobel(dx)
dxy, dyy = ops.sobel(dy)
filtered = sift.filter_keypoints(doG_i, keypoints, dxx, dyy, dxy, r=sift.r)
print("Số keypoint ứng viên:", len(keypoints), "| sau khi lọc:", len(filtered))
fig, ax = plt.subplots(1, 2, figsize=(16, 5))
for a, kps, t in [(ax[0], keypoints, "Keypoint ứng viên"), (ax[1], filtered, "Sau khi lọc cạnh")]:
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
    if kps:
        k = np.array(kps); a.scatter(k[:, 0], k[:, 1], s=6, c="red")
plt.show()

## Bước 5 – Gán hướng chính
Histogram 36 bin (mỗi bin 10°) của hướng gradient trong vùng 8×8, trọng số là độ lớn gradient. Bin cao nhất là hướng chính; các bin ≥ 80% đỉnh là hướng phụ.

In [ ]:
mag, ori = ops.cart_to_polar(dx, dy)
orientation_data, kps = sift.get_dominant_angles(filtered, mag, ori)
plt.figure(figsize=(8, 6)); plt.imshow(img, cmap="gray"); plt.axis("off")
for (x, y) in kps:
    for a in orientation_data[(x, y)]:
        plt.arrow(x, y, 12 * np.cos(np.deg2rad(a)), 12 * np.sin(np.deg2rad(a)), color="yellow", head_width=3)
plt.title("Hướng chính của các keypoint"); plt.show()

## Bước 6 – Descriptor 128 chiều
Vùng 16×16 quanh điểm → xoay theo hướng chính → chia 4×4 ô → mỗi ô histogram 8 hướng → 16 × 8 = 128 số → chuẩn hóa độ dài 1.

In [ ]:
kp0 = kps[0]; a0 = orientation_data[kp0][0]
d0 = sift.get_descriptor(doG_i, kp0, a0)
plt.figure(figsize=(12, 3)); plt.bar(range(len(d0)), d0)
plt.title(f"Descriptor của keypoint {kp0} (hướng {a0:.0f}°) – {len(d0)} chiều"); plt.show()

## Bước 7 – Kiểm tra bất biến tỉ lệ và phép xoay
So khớp SIFT giữa ảnh gốc và ảnh đã **thu nhỏ 0.8 lần** / **xoay 20°**, rồi vẽ các cặp điểm khớp (đã qua RANSAC).

In [ ]:
def match_and_plot(img1, img2, title):
    s = SIFT(num_downsamples=0)
    f1, f2 = s.extract_sift_data(img1), s.extract_sift_data(img2)
    i1, i2, _ = ops.match_descriptors(np.stack(f1["Descriptor"]), np.stack(f2["Descriptor"]), 1.0, 0.8)
    p1 = np.array(f1["Position"].to_list(), float)[i1]
    p2 = np.array(f2["Position"].to_list(), float)[i2]
    H, inl = ops.ransac_homography(p1, p2, 5.0)
    print(f"{title}: {len(f1)} / {len(f2)} keypoint, {len(i1)} cặp khớp, {inl.sum()} inlier")
    h = max(img1.shape[0], img2.shape[0])
    canvas = np.zeros((h, img1.shape[1] + img2.shape[1]))
    canvas[:img1.shape[0], :img1.shape[1]] = img1
    canvas[:img2.shape[0], img1.shape[1]:] = img2
    plt.figure(figsize=(14, 6)); plt.imshow(canvas, cmap="gray"); plt.axis("off")
    for a, b in zip(p1[inl], p2[inl]):
        plt.plot([a[0], b[0] + img1.shape[1]], [a[1], b[1]], "-", lw=0.6, color="lime")
    plt.title(title); plt.show()

match_and_plot(img, ops.resize_bilinear(img, 0.8), "Bất biến tỉ lệ (ảnh thu nhỏ 0.8)")
match_and_plot(img, ops.rotate(img, 20), "Bất biến phép xoay (ảnh xoay 20°)")